In [1]:
using JuMP
using HiGHS
# using EzXML
using Plots
# using Ipopt
# using Cbc
import MultiObjectiveAlgorithms as MOA
import MathOptInterface as MOI
using Printf
using PrettyTables

In [2]:
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7

7

In [3]:
function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
        Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
        Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
        Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
        Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
        w = parse.(Int, split(lines[idx])); idx += 1
        b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        v = parse.(Int, split(lines[idx])); idx += 1
        V = parse.(Int, split(lines[idx])); idx += 1

        B_star = Array{Int, 3}(undef, S, P, length(D_star))
        for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        C_star = Array{Int, 3}(undef, T, P, length(D_star))
        for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        if validate==true
                # input validation


        end

        return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star
end

read_input (generic function with 1 method)

In [4]:
function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

ρ (generic function with 1 method)

In [5]:
# Get input parameters from .txt file
input_files_folder = "input"
# input_file_name = "input_smaller.txt"
input_file_name = "input_current.txt"
file_path = input_files_folder * "/" * input_file_name
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

(56, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  47, 48, 49, 50, 51, 52, 53, 54, 55, 56], [6, 13, 20, 27, 34, 41, 48, 55], 18, SubString{String}["Player1", "Player2", "Player3", "Player4", "Player5", "Player6", "Player7", "Player8", "Player9", "Player10", "Player11", "Player12", "Player13", "Player14", "Player15", "Player16", "Player17", "Player18"], 38, SubString{String}["Ускорение", "Торможение", "Реакция", "ЛовляНаУрТуловища", "СменаХвата", "ПриставнойШаг", "СменаНаправления", "Лестница", "ПрофилактическиеУпражнения", "БеговыеУпражнения"  …  "Хаммер5Слабый", "Блейд5Сильный", "Хаммер10Сильный", "Блейд10Сильный", "Скубер10Сильный", "ФорхендБэкхенд25Вышагивание", "ФорхендБэкхенд50", "ХаммерБлейдСкубер25", "ПеревернутыйБэкхенд25", "ОбводнойБэкхенд25"], 10, SubString{String}["Зоны", "Роли", "ВертикальныйСтек", "ЛичнаяЗащита", "ГоризонтальныйСтек", "ДиагональныйСтек", "РазделенныйСтек", "СменаИПодстраховка", "АтакаВблизиГола", "СтандартныеПозиционныеСхемы"], 48, SubString{String}["Ускорение_1", "Т

In [6]:
# Define model
model = Model(HiGHS.Optimizer)
# model = Model(() -> MOA.Optimizer(
#         optimizer_with_attributes(
#         HiGHS.Optimizer,
#         "output_flag" => true,
#         "mip_rel_gap" => 0.05,
#         "threads" => 8
#         )
# ))

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [7]:
# Define variables
@variable(model, x[1:E, 1:length(D_)] >= 0, Int)

48×56 Matrix{VariableRef}:
 x[1,1]   x[1,2]   x[1,3]   x[1,4]   x[1,5]   …  x[1,54]   x[1,55]   x[1,56]
 x[2,1]   x[2,2]   x[2,3]   x[2,4]   x[2,5]      x[2,54]   x[2,55]   x[2,56]
 x[3,1]   x[3,2]   x[3,3]   x[3,4]   x[3,5]      x[3,54]   x[3,55]   x[3,56]
 x[4,1]   x[4,2]   x[4,3]   x[4,4]   x[4,5]      x[4,54]   x[4,55]   x[4,56]
 x[5,1]   x[5,2]   x[5,3]   x[5,4]   x[5,5]      x[5,54]   x[5,55]   x[5,56]
 x[6,1]   x[6,2]   x[6,3]   x[6,4]   x[6,5]   …  x[6,54]   x[6,55]   x[6,56]
 x[7,1]   x[7,2]   x[7,3]   x[7,4]   x[7,5]      x[7,54]   x[7,55]   x[7,56]
 x[8,1]   x[8,2]   x[8,3]   x[8,4]   x[8,5]      x[8,54]   x[8,55]   x[8,56]
 x[9,1]   x[9,2]   x[9,3]   x[9,4]   x[9,5]      x[9,54]   x[9,55]   x[9,56]
 x[10,1]  x[10,2]  x[10,3]  x[10,4]  x[10,5]     x[10,54]  x[10,55]  x[10,56]
 ⋮                                            ⋱                      ⋮
 x[40,1]  x[40,2]  x[40,3]  x[40,4]  x[40,5]     x[40,54]  x[40,55]  x[40,56]
 x[41,1]  x[41,2]  x[41,3]  x[41,4]  x[41,5]  …  x[41

In [8]:
@variable(model, y[1:S, 1:P, 1:length(D_star)], Bin)

38×18×8 Array{VariableRef, 3}:
[:, :, 1] =
 y[1,1,1]   y[1,2,1]   y[1,3,1]   …  y[1,16,1]   y[1,17,1]   y[1,18,1]
 y[2,1,1]   y[2,2,1]   y[2,3,1]      y[2,16,1]   y[2,17,1]   y[2,18,1]
 y[3,1,1]   y[3,2,1]   y[3,3,1]      y[3,16,1]   y[3,17,1]   y[3,18,1]
 y[4,1,1]   y[4,2,1]   y[4,3,1]      y[4,16,1]   y[4,17,1]   y[4,18,1]
 y[5,1,1]   y[5,2,1]   y[5,3,1]      y[5,16,1]   y[5,17,1]   y[5,18,1]
 y[6,1,1]   y[6,2,1]   y[6,3,1]   …  y[6,16,1]   y[6,17,1]   y[6,18,1]
 y[7,1,1]   y[7,2,1]   y[7,3,1]      y[7,16,1]   y[7,17,1]   y[7,18,1]
 y[8,1,1]   y[8,2,1]   y[8,3,1]      y[8,16,1]   y[8,17,1]   y[8,18,1]
 y[9,1,1]   y[9,2,1]   y[9,3,1]      y[9,16,1]   y[9,17,1]   y[9,18,1]
 y[10,1,1]  y[10,2,1]  y[10,3,1]     y[10,16,1]  y[10,17,1]  y[10,18,1]
 ⋮                                ⋱  ⋮                       
 y[30,1,1]  y[30,2,1]  y[30,3,1]     y[30,16,1]  y[30,17,1]  y[30,18,1]
 y[31,1,1]  y[31,2,1]  y[31,3,1]  …  y[31,16,1]  y[31,17,1]  y[31,18,1]
 y[32,1,1]  y[32,2,1]  y[32,3,1]     y[3

In [9]:
@variable(model, z[1:T, 1:P, 1:length(D_star)], Bin)

10×18×8 Array{VariableRef, 3}:
[:, :, 1] =
 z[1,1,1]   z[1,2,1]   z[1,3,1]   …  z[1,16,1]   z[1,17,1]   z[1,18,1]
 z[2,1,1]   z[2,2,1]   z[2,3,1]      z[2,16,1]   z[2,17,1]   z[2,18,1]
 z[3,1,1]   z[3,2,1]   z[3,3,1]      z[3,16,1]   z[3,17,1]   z[3,18,1]
 z[4,1,1]   z[4,2,1]   z[4,3,1]      z[4,16,1]   z[4,17,1]   z[4,18,1]
 z[5,1,1]   z[5,2,1]   z[5,3,1]      z[5,16,1]   z[5,17,1]   z[5,18,1]
 z[6,1,1]   z[6,2,1]   z[6,3,1]   …  z[6,16,1]   z[6,17,1]   z[6,18,1]
 z[7,1,1]   z[7,2,1]   z[7,3,1]      z[7,16,1]   z[7,17,1]   z[7,18,1]
 z[8,1,1]   z[8,2,1]   z[8,3,1]      z[8,16,1]   z[8,17,1]   z[8,18,1]
 z[9,1,1]   z[9,2,1]   z[9,3,1]      z[9,16,1]   z[9,17,1]   z[9,18,1]
 z[10,1,1]  z[10,2,1]  z[10,3,1]     z[10,16,1]  z[10,17,1]  z[10,18,1]

[:, :, 2] =
 z[1,1,2]   z[1,2,2]   z[1,3,2]   …  z[1,16,2]   z[1,17,2]   z[1,18,2]
 z[2,1,2]   z[2,2,2]   z[2,3,2]      z[2,16,2]   z[2,17,2]   z[2,18,2]
 z[3,1,2]   z[3,2,2]   z[3,3,2]      z[3,16,2]   z[3,17,2]   z[3,18,2]
 z[4,1,2]   z[4,2,2]

In [10]:
@variable(model, Z[1:T, 1:length(D_star)], Bin)

10×8 Matrix{VariableRef}:
 Z[1,1]   Z[1,2]   Z[1,3]   Z[1,4]   Z[1,5]   Z[1,6]   Z[1,7]   Z[1,8]
 Z[2,1]   Z[2,2]   Z[2,3]   Z[2,4]   Z[2,5]   Z[2,6]   Z[2,7]   Z[2,8]
 Z[3,1]   Z[3,2]   Z[3,3]   Z[3,4]   Z[3,5]   Z[3,6]   Z[3,7]   Z[3,8]
 Z[4,1]   Z[4,2]   Z[4,3]   Z[4,4]   Z[4,5]   Z[4,6]   Z[4,7]   Z[4,8]
 Z[5,1]   Z[5,2]   Z[5,3]   Z[5,4]   Z[5,5]   Z[5,6]   Z[5,7]   Z[5,8]
 Z[6,1]   Z[6,2]   Z[6,3]   Z[6,4]   Z[6,5]   Z[6,6]   Z[6,7]   Z[6,8]
 Z[7,1]   Z[7,2]   Z[7,3]   Z[7,4]   Z[7,5]   Z[7,6]   Z[7,7]   Z[7,8]
 Z[8,1]   Z[8,2]   Z[8,3]   Z[8,4]   Z[8,5]   Z[8,6]   Z[8,7]   Z[8,8]
 Z[9,1]   Z[9,2]   Z[9,3]   Z[9,4]   Z[9,5]   Z[9,6]   Z[9,7]   Z[9,8]
 Z[10,1]  Z[10,2]  Z[10,3]  Z[10,4]  Z[10,5]  Z[10,6]  Z[10,7]  Z[10,8]

In [11]:
@variable(model, A_min[1:length(D_star)])

8-element Vector{VariableRef}:
 A_min[1]
 A_min[2]
 A_min[3]
 A_min[4]
 A_min[5]
 A_min[6]
 A_min[7]
 A_min[8]

In [12]:
# Вспомогательные функции и компоненты целевой функции
function A(p::Int, d::Int)
	result = a0[p]
	result += sum(δ1(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	result -= sum(δ2(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function A_avg(d::Int)
	return sum(A(p, d) for p in 1:P)/P
end

α = 0.5
function A()
	return sum(α*A_min[d_]+(1-α)*A_avg(D_star[d_]) for d_ in 1:length(D_star))
end

function B(s::Int, p::Int, d::Int)
	result = b0[s][p]
	result += sum(h[p][d_]*sum(b[s][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function B()
	return sum(sum(sum(β(s, p, d_)*y[s, p, d_] for s in 1:S) for p in 1:P) for d_ in 1:length(D_star))
end

function C(t::Int, p::Int, d::Int)
	result = c0[t][p]
	result += sum(h[p][d_]*sum(c[t][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
end

function C()
	return sum(sum(γ(t, d_)*Z[t, d_] for t in 1:T) for d_ in 1:length(D_star))
end

C (generic function with 2 methods)

In [13]:
# Define constraints
@constraint(model, [d_=1:length(D_)], sum(v[e]*x[e, d_] for e in 1:E) <= V[d_])

56-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 x[1,1] + 2 x[2,1] + x[3,1] + x[4,1] + 2 x[5,1] + x[6,1] + x[7,1] + x[8,1] + 2 x[9,1] + x[10,1] + x[11,1] + 2 x[12,1] + x[13,1] + x[14,1] + 2 x[15,1] + x[16,1] + x[17,1] + x[18,1] + 2 x[19,1] + x[20,1] + x[21,1] + 2 x[22,1] + x[23,1] + x[24,1] + 2 x[25,1] + x[26,1] + x[27,1] + x[28,1] + 2 x[29,1] + x[30,1] + x[31,1] + 2 x[32,1] + x[33,1] + x[34,1] + 2 x[35,1] + x[36,1] + x[37,1] + x[38,1] + 2 x[39,1] + x[40,1] + x[41,1] + 2 x[42,1] + x[43,1] + x[44,1] + 2 x[45,1] + x[46,1] + x[47,1] + x[48,1] <= 18
 x[1,2] + 2 x[2,2] + x[3,2] + x[4,2] + 2 x[5,2] + x[6,2] + x[7,2] + x[8,2] + 2 x[9,2] + x[10,2] + x[11,2] + 2 x[12,2] + x[13,2] + x[14,2] + 2 x[15,2] + x[16,2] + x[17,2] + x[18,2] + 2 x[19,2] + x[20,2] + x[21,2] + 2 x[22,2] + x[23,2] + x[24,2] + 2 x[25,2] + x[26,2] + x[27,2] + x[28,2] + 2 x[29,2] + x[30,2] + x[31,2] + 2 x[

In [14]:
@constraint(model, [s=1:S, p=1:P, d_=1:length(D_star)], B(s, p, D_star[d_]) + B_star[s, p, d_]*(1-y[s, p, d_]) >= B_star[s, p, d_])

38×18×8 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 x[1,1] + x[1,2] + x[1,3] + x[1,4] + x[1,5] + x[1,6] - 10 y[1,1,1] >= -5         …  x[1,1] + x[1,2] + x[1,3] + x[1,4] + x[1,5] + x[1,6] - 10 y[1,18,1] >= -5
 x[2,1] + x[2,2] + x[2,3] + x[2,4] + x[2,5] + x[2,6] - 10 y[2,1,1] >= -8            x[2,1] + x[2,2] + x[2,3] + x[2,4] + x[2,5] + x[2,6] - 10 y[2,18,1] >= -9
 x[3,1] + x[3,2] + x[3,3] + x[3,4] + x[3,5] + x[3,6] - 10 y[3,1,1] >= -4            x[3,1] + x[3,2] + x[3,3] + x[3,4] + x[3,5] + x[3,6] - 10 y[3,18,1] >= -6
 x[4,1] + x[4,2] + x[4,3] + x[4,4] + x[4,5] + x[4,6] - 10 y[4,1,1] >= -5            x[4,1] + x[4,2] + x[4,3] + x[4,4] + x[4,5] + x[4,6] - 10 y[4,18,1] >= -7
 x[5,1] + x[5,2] + x[5,3] + x[5,4] + x[5,5] + x[5,6] - 10 y[5,1,1] >= -8            x[5,1] + x[5,2] + x[5,3] + x[5,4] + x[5,5] + x[5,6] - 10 y[5,18,1] >= -6
 x[6,1] + x[6,2] + x[6,3] + x

In [15]:
@constraint(model, [t=1:T, p=1:P, d_=1:length(D_star)], C(t, p, D_star[d_]) + C_star[t, p, d_]*(1-z[t, p, d_]) >= C_star[t, p, d_])

10×18×8 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 x[39,1] + x[39,2] + x[39,3] + x[39,4] + x[39,5] + x[39,6] - 10 z[1,1,1] >= -5   …  x[39,1] + x[39,2] + x[39,3] + x[39,4] + x[39,5] + x[39,6] - 10 z[1,18,1] >= -6
 x[40,1] + x[40,2] + x[40,3] + x[40,4] + x[40,5] + x[40,6] - 10 z[2,1,1] >= -8      x[40,1] + x[40,2] + x[40,3] + x[40,4] + x[40,5] + x[40,6] - 10 z[2,18,1] >= -6
 x[41,1] + x[41,2] + x[41,3] + x[41,4] + x[41,5] + x[41,6] - 10 z[3,1,1] >= -4      x[41,1] + x[41,2] + x[41,3] + x[41,4] + x[41,5] + x[41,6] - 10 z[3,18,1] >= -7
 x[42,1] + x[42,2] + x[42,3] + x[42,4] + x[42,5] + x[42,6] - 10 z[4,1,1] >= -5      x[42,1] + x[42,2] + x[42,3] + x[42,4] + x[42,5] + x[42,6] - 10 z[4,18,1] >= -6
 x[43,1] + x[43,2] + x[43,3] + x[43,4] + x[43,5] + x[43,6] - 10 z[5,1,1] >= -8      x[43,1] + x[43,2] + x[43,3] + x[43,4] + x[43,5] + x[43,6] - 10 z[5,18,1] >= -6

In [16]:
@constraint(model, [t=1:T, d_=1:length(D_star)], sum(z[t, p, d_] for p in 1:P) >= ρ(t)*Z[t, d_])

10×8 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}}:
 z[1,1,1] + z[1,2,1] + z[1,3,1] + z[1,4,1] + z[1,5,1] + z[1,6,1] + z[1,7,1] + z[1,8,1] + z[1,9,1] + z[1,10,1] + z[1,11,1] + z[1,12,1] + z[1,13,1] + z[1,14,1] + z[1,15,1] + z[1,16,1] + z[1,17,1] + z[1,18,1] - 2 Z[1,1] >= 0                     …  z[1,1,8] + z[1,2,8] + z[1,3,8] + z[1,4,8] + z[1,5,8] + z[1,6,8] + z[1,7,8] + z[1,8,8] + z[1,9,8] + z[1,10,8] + z[1,11,8] + z[1,12,8] + z[1,13,8] + z[1,14,8] + z[1,15,8] + z[1,16,8] + z[1,17,8] + z[1,18,8] - 2 Z[1,8] >= 0
 z[2,1,1] + z[2,2,1] + z[2,3,1] + z[2,4,1] + z[2,5,1] + z[2,6,1] + z[2,7,1] + z[2,8,1] + z[2,9,1] + z[2,10,1] + z[2,11,1] + z[2,12,1] + z[2,13,1] + z[2,14,1] + z[2,15,1] + z[2,16,1] + z[2,17,1] + z[2,18,1] - 2 Z[2,1] >= 0                        z[2,1,8] + z[2,2,8] + z[2,3,8] + z[2,4,8] + z[2,5,8] + z[2,6,8] + z[2,7,8] + z[2,8,8] + z[2,9,8] + z[2,10,8] + z[2,11,8]

In [17]:
@constraint(model, [d_=1:length(D_star), p=1:P], A_min[d_] <= A(p, D_star[d_]))

8×18 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 0.09131779390732842 x[1,1] + 0.18263558781465683 x[2,1] + 0.27395338172198525 x[3,1] + 0.36527117562931366 x[4,1] + 0.4565889695366421 x[5,1] + 0.09131779390732842 x[6,1] + 0.18263558781465683 x[7,1] + 0.27395338172198525 x[8,1] + 0.36527117562931366 x[9,1] + 0.4565889695366421 x[10,1] + 0.09131779390732842 x[11,1] + 0.18263558781465683 x[12,1] + 0.27395338172198525 x[13,1] + 0.36527117562931366 x[14,1] + 0.4565889695366421 x[15,1] + 0.09131779390732842 x[16,1] + 0.18263558781465683 x[17,1] + 0.27395338172198525 x[18,1] + 0.36527117562931366 x[19,1] + 0.4565889695366421 x[20,1] + 0.09131779390732842 x[21,1] + 0.18263558781465683 x[22,1] + 0.27395338172198525 x[23,1] + 0.36527117562931366 x[24,1] + 0.4565889695366421 x[25,1] + 0.09131779390732842 x[26,1] + 0.18263558781465683 x[27,1] + 0.27395338172198525 x[28,1] + 0.36527

In [18]:
# Get ideal point
@objective(model, Max, A())
set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
set_optimizer_attribute(model, "mip_rel_gap", 0.01)
set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes
optimize!(model)
A_optimal = value(A())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

Running HiGHS 1.13.0 (git hash: 1bce6d5c8): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
MIP has 7192 rows; 9688 cols; 383360 nonzeros; 9680 integer variables (6992 binary)
Coefficient ranges:
  Matrix  [2e-02, 8e+01]
  Cost    [6e-03, 7e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
7191 rows, 4088 cols, 377760 nonzeros  0s
152 rows, 302 cols, 19543 nonzeros  0s
138 rows, 187 cols, 11327 nonzeros  0s
Presolve reductions: rows 138(-7054); columns 187(-9501); nonzeros 11327(-372033) 

Solving MIP model with:
   138 rows
   187 cols (0 binary, 180 integer, 0 implied int., 7 continuous, 0 domain fixed)
   11327 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lo

In [19]:
@objective(model, Max, B())
set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
set_optimizer_attribute(model, "mip_rel_gap", 0.1)
set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes
optimize!(model)
B_optimal = value(B())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

MIP has 7192 rows; 9688 cols; 383360 nonzeros; 9680 integer variables (6992 binary)
Coefficient ranges:
  Matrix  [2e-02, 8e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
7047 rows, 9552 cols, 197040 nonzeros  0s
5527 rows, 7562 cols, 154850 nonzeros  0s
Presolve reductions: rows 5527(-1665); columns 7562(-2126); nonzeros 154850(-228510) 
Objective function is integral with scale 1

Solving MIP model with:
   5527 rows
   7562 cols (5472 binary, 2090 integer, 0 implied int., 0 continuous, 0 domain fixed)
   154850 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |    

In [20]:
@objective(model, Max, C())
set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
set_optimizer_attribute(model, "mip_rel_gap", 0.01)
set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes
optimize!(model)
C_optimal = value(C())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

MIP has 7192 rows; 9688 cols; 383360 nonzeros; 9680 integer variables (6992 binary)
Coefficient ranges:
  Matrix  [2e-02, 8e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
7047 rows, 4160 cols, 191648 nonzeros  0s
1575 rows, 2070 cols, 42270 nonzeros  0s
Presolve reductions: rows 1575(-5617); columns 2070(-7618); nonzeros 42270(-341090) 
Objective function is integral with scale 1

Solving MIP model with:
   1575 rows
   2070 cols (1520 binary, 550 integer, 0 implied int., 0 continuous, 0 domain fixed)
   42270 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |        

In [21]:
ideal_point = [A_optimal, B_optimal, C_optimal]

3-element Vector{Float64}:
 4652.735532049485
 1276.0
   80.0

In [22]:
A_norm = () -> A()/A_optimal
B_norm = () -> B()/B_optimal
C_norm = () -> C()/C_optimal

#91 (generic function with 1 method)

In [23]:
# Create an array of weights
weights_for_moa =   [[0.33, 0.33, 0.33],
                     [0.5, 0.25, 0.25],
                     [0.25, 0.5, 0.25],
                     [0.25, 0.25, 0.5],
                    ]

4-element Vector{Vector{Float64}}:
 [0.33, 0.33, 0.33]
 [0.5, 0.25, 0.25]
 [0.25, 0.5, 0.25]
 [0.25, 0.25, 0.5]

In [24]:
# Solve model with different weight distributions using warm-start (progressive gap tightening)
set_silent(model)

n_passes = 3                      # warm-start passes per weight combination
gaps = [0.15, 0.10, 0.05]         # progressively tighter MIP relative gaps

output_folder = "results"
mkpath(output_folder)
solutions = []
solutions_nonrm = []
solution_weights = []   # weights for each successful solution (for visualization)

for i in 1:length(weights_for_moa)
    wA, wB, wC = weights_for_moa[i]
    @objective(model, Max, wA*A_norm() + wB*B_norm() + wC*C_norm())

    println("Solving weight combination $i/$(length(weights_for_moa)): wA=$wA, wB=$wB, wC=$wC")

    # best_sol stores all values extracted right after optimize!(), before any model modification
    best_sol = nothing

    for pass in 1:n_passes
        set_optimizer_attribute(model, "mip_rel_gap", gaps[pass])
		set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes
        optimize!(model)

        if primal_status(model) != MOI.FEASIBLE_POINT
            println("  Pass $pass: no feasible solution (status=$(termination_status(model)))")
            break
        end

        println("  Pass $pass (gap=$(gaps[pass])): obj = $(round(objective_value(model), digits=6))")
		if termination_status(model) == TIME_LIMIT
			gap = relative_gap(model)
			println("Relative gap: $(gap * 100)%")
		end

        # ── Extract ALL values immediately, before any model modification ──
        x_vals     = round.(Int, value.(x))
        y_vals     = round.(Int, value.(y))
        z_vals     = round.(Int, value.(z))
        Z_vals     = round.(Int, value.(Z))
        A_min_vals = value.(A_min)
        # Compute objective components while model is still in solved state
        A_val      = value(A())
        B_val      = value(B())
        C_val      = value(C())
        An_val     = value(A_norm())
        Bn_val     = value(B_norm())
        Cn_val     = value(C_norm())

        best_sol = (x=x_vals, y=y_vals, z=z_vals, Z=Z_vals, A_min=A_min_vals,
                    A=A_val, B=B_val, C=C_val,
                    A_norm=An_val, B_norm=Bn_val, C_norm=Cn_val)

        # ── Now safe to modify model for warm-start ──
        if pass < n_passes
            set_start_value.(x,     x_vals)
            set_start_value.(y,     y_vals)
            set_start_value.(z,     z_vals)
            set_start_value.(Z,     Z_vals)
            set_start_value.(A_min, A_min_vals)
        end
    end

    if best_sol === nothing
        println("  Weight combination $i: no feasible solution found, skipping.\n")
        continue
    end

    push!(solutions,        [best_sol.A_norm, best_sol.B_norm, best_sol.C_norm])
    push!(solutions_nonrm,  [best_sol.A,      best_sol.B,      best_sol.C])
    push!(solution_weights, [wA, wB, wC])

    output_file = "$(input_file_name)_$(weights_for_moa[i])"

    open(output_folder * "/" * output_file, "w") do io
        println(io, "Solution of model with weight distribution ", weights_for_moa[i])
        println(io, "Ideal Point (Bound): ", ideal_point)

        println(io, "---Итоговая эффективность тренировок---")
        println(io, "A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
        println(io, "A = $(best_sol.A)\nB = $(best_sol.B)\nC = $(best_sol.C)\n\n")
        println(io, "---Итоговая эффективность тренировок относительно идеальной точки---")
        println(io, "A = $(best_sol.A_norm)\nB = $(best_sol.B_norm)\nC = $(best_sol.C_norm)\n\n")

        println(io, "---Расписание тренировок---\n")
        n_days = length(D_)
        data = Array{Any}(undef, n_days, 2)
        for d_ in 1:n_days
            data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"
            day_exercises = String[]
            for e in 1:E
                if best_sol.x[e, d_] >= 1
                    push!(day_exercises, Exercise_names[e])
                end
            end
            data[d_, 2] = join(day_exercises, ", ")
        end
        pretty_table(io, data)

        println(io, "---Освоенные навыки---\n")
        for p in 1:P
            for d_ in 1:length(D_star)
                println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                learned_anything = false
                for s in 1:S
                    if best_sol.y[s, p, d_] == 1 && (d_ == 1 || best_sol.y[s, p, d_-1] != 1)
                        learned_anything = true
                        println(io, "Освоил навык \'$(Skill_names[s])\'")
                    end
                end
                if !learned_anything; println(io, "Ничего не освоил"); end
            end
            print(io, "\n")
        end

        println(io, "---Освоенные индивидуально тактики---\n")
        for p in 1:P
            for d_ in 1:length(D_star)
                println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                learned_anything = false
                for t in 1:T
                    if best_sol.z[t, p, d_] == 1 && (d_ == 1 || best_sol.z[t, p, d_-1] != 1)
                        learned_anything = true
                        println(io, "Освоил тактику \'$(Tactic_names[t])\'")
                    end
                end
                if !learned_anything; println(io, "Ничего не освоил"); end
            end
            print(io, "\n")
        end

        println(io, "---Освоенные командой тактики---\n")
        for d_ in 1:length(D_star)
            for t in 1:T
                println(io, "Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
                println(io, best_sol.Z[t, d_] == 1 ? "Освоено" : "Не освоено")
            end
            print(io, "\n")
        end
    end

    println()
end


Solving weight combination 1/4: wA=0.33, wB=0.33, wC=0.33
  Pass 1 (gap=0.15): obj = 0.552139
Relative gap: 77.68746992449712%
  Pass 2 (gap=0.1): obj = 0.57345
Relative gap: 69.18075035019586%
  Pass 3 (gap=0.05): obj = 0.576568
Relative gap: 68.25616260582488%

Solving weight combination 2/4: wA=0.5, wB=0.25, wC=0.25
  Pass 1 (gap=0.15): obj = 0.648947
Relative gap: 44.82575333281923%
  Pass 2 (gap=0.1): obj = 0.648947
Relative gap: 44.108735838189105%
  Pass 3 (gap=0.05): obj = 0.66591
Relative gap: 40.68132302557505%

Solving weight combination 3/4: wA=0.25, wB=0.5, wC=0.25
  Pass 1 (gap=0.15): obj = 0.620344
Relative gap: 68.24135891617108%
  Pass 2 (gap=0.1): obj = 0.620344
Relative gap: 68.6941498651525%
  Pass 3 (gap=0.05): obj = 0.640915
Relative gap: 63.775508421989144%

Solving weight combination 4/4: wA=0.25, wB=0.25, wC=0.5
  Pass 1 (gap=0.15): obj = 0.522888
Relative gap: 90.92390146702803%
  Pass 2 (gap=0.1): obj = 0.524575
Relative gap: 87.39480080974872%
  Pass 3 (gap=

In [25]:
# Write visualization data to txt file for Python visualizer
viz_file = output_folder * "/" * input_file_name * "_viz_data.txt"
open(viz_file, "w") do io
    println(io, "# Multi-criteria optimization visualization data")
    println(io, "# ideal_point,A,B,C")
    println(io, "# solution,wA,wB,wC,A,B,C,A_norm,B_norm,C_norm")
    println(io, "ideal_point,$(ideal_point[1]),$(ideal_point[2]),$(ideal_point[3])")
    for i in 1:length(solutions)
        wA, wB, wC = solution_weights[i]
        A_val, B_val, C_val = solutions_nonrm[i]
        An, Bn, Cn = solutions[i]
        println(io, "solution,$wA,$wB,$wC,$A_val,$B_val,$C_val,$An,$Bn,$Cn")
    end
end
println("Visualization data written to: $viz_file")


Visualization data written to: results/input_current.txt_viz_data.txt
